# How the spot-focus edge point is picked

The spot focus lock follows one straight edge of the LED patch. This notebook
shows, step by step, how an **edge box** on the camera image turns into one
number (the edge position in sensor pixels) and the cross drawn on the live
view. It also compares the two detectors that are built in
(`halfmax` and `gaussderiv`) with the Sobel, Prewitt and Scharr filters, to
explain why **Gaussian derivative** is the default and the classic filters are
not used.

It runs top to bottom. If the rig calibration sweeps
(`spot_calibration_*.npz` in the app data folder) are present they are used,
otherwise a synthetic patch with a soft, asymmetric left edge is built so that
everything still runs. Plots need `matplotlib` (not a project dependency;
`pip install matplotlib`). The tables are printed either way.

Companion text: `docs/spot_focus_lock.md`, section "How the edge point is
picked".

In [ ]:
# ruff: noqa: D103
import json
import math
import os
from pathlib import Path

import numpy as np

from pymmcore_gui.asi_z_stack.spot_focus import measure_edge, snap_box
from pymmcore_gui.asi_z_stack.spot_synthetic import add_noise, render_led_patch

try:
    import matplotlib.pyplot as plt
except ImportError:  # the tables are printed either way
    plt = None
    print("matplotlib is not installed: plots are skipped.")
    print("pip install matplotlib to see them.")

DETECTORS = ("halfmax", "gaussderiv", "sobel", "prewitt", "scharr")
SIGMA_BIN = 3.0  # gaussderiv width: sigma_px 6 sensor px = 3 binned px

## 1. The data

A calibration sweep moves piezo P through 9 positions (about +-1 um) and keeps
one camera frame per step. The top edge of the patch is **box 1** (normal along
y), the left edge is **box 2** (normal along x). The saved frames are the
camera ROI (`438 x 538` here); the ROI's sensor offset comes from the app
settings, so box coordinates are sensor pixels.

Rig data stay in the app data folder and are never copied into the repo. With
no rig data the notebook renders a synthetic patch: a top edge with a normal
Gaussian blur and a left edge with a soft, asymmetric ramp (bright rim, then a
shoulder), the kind of profile seen on the rig.

In [ ]:
DATA_DIR = (
    Path(os.environ.get("LOCALAPPDATA", str(Path.home())))
    / "pymmcore-gui"
    / "pymmcore-gui"
)


def saved_roi():
    # Sensor (x, y) of the first pixel of the saved frames: the ROI in the app settings.
    try:
        roi = json.loads((DATA_DIR / "pmm_settings.json").read_text())["spot_focus"][
            "roi"
        ]
        return int(roi[0]), int(roi[1])
    except (OSError, KeyError, IndexError, TypeError, ValueError):
        return 610, 562  # the ROI of the 2026-10-28 rig sessions


def load_rig_sweep(path):
    # One calibration npz (two-box layout, or the older single-box one).
    z = np.load(path, allow_pickle=True)
    if "boxes" in z.files:
        boxes = [tuple(int(v) for v in b) for b in z["boxes"]]
        axes = [str(a) for a in z["axes"]]
        pols = [int(p) for p in z["polarities"]]
    else:
        boxes = [tuple(int(v) for v in z["box"])]
        axes, pols = [str(z["axis"])], [int(z["polarity"])]
    return {
        "name": path.stem.removeprefix("spot_calibration_"),
        "p": np.asarray(z["p_um"], dtype=float),
        "frames": np.asarray(z["frames"]),
        "roi": saved_roi(),
        "boxes": boxes,
        "axes": axes,
        "pols": pols,
    }


def ramp_profile(x, left):
    # Asymmetric left edge: a steep rise (72 %) then a shoulder that creeps up.
    erf = np.vectorize(math.erf)
    phi = lambda z: 0.5 * (1.0 + erf(z / math.sqrt(2.0)))  # noqa: E731
    return 0.72 * phi((x - left) / 6.0) + 0.28 * phi((x - left - 16.0) / 9.0)


def synthetic_sweep(seed, slopes=(2.0, 1.7), noise=0.4):
    # Nine 8-bit frames of an LED patch whose top / left edges move with P.
    rng = np.random.default_rng(seed)
    p = np.linspace(-1.0, 1.0, 9)
    shape = (438, 538)
    x = np.r_[: shape[1]].astype(float)
    frames = []
    for pk in p:
        left, top = 199.0 + slopes[1] * pk, 36.0 + slopes[0] * pk
        x_l, x_r, y_b = left - 40.0, 499.0, 336.0  # the left rim is hidden by the ramp
        raw = render_led_patch(
            shape,
            ((x_l + x_r) / 2.0, (top + y_b) / 2.0),
            size=(x_r - x_l, y_b - top),
            edge_sigma=3.0,
            level=27.0,
            texture=0.03,
            tail=0.0,
            ghost=0.0,
            seed=seed,
        )
        img = 1.0 + (raw - 1.0) * ramp_profile(x, left)[None, :]
        frames.append(add_noise(img, noise=noise, rng=rng, bits=8))
    return {
        "name": f"synthetic{seed}",
        "p": p,
        "frames": np.array(frames),
        "roi": (610, 562),
        "boxes": [(880, 580, 106, 36), (790, 660, 38, 66)],
        "axes": ["y", "x"],
        "pols": [1, 1],
    }


rig_paths = sorted(DATA_DIR.glob("spot_calibration_*.npz"))
# 132339 is a very noisy single-edge sweep; it is left out as in the detector study.
SWEEPS = [load_rig_sweep(p) for p in rig_paths if "132339" not in p.name]
two_box = [s for s in SWEEPS if len(s["boxes"]) == 2]
if two_box:
    SOURCE, DEMO = "rig data", two_box[-1]
else:
    SOURCE = "synthetic patch"
    SWEEPS = [synthetic_sweep(seed) for seed in range(4)]
    DEMO = SWEEPS[-1]
K = 1  # the demo box: the left edge (box 2)
print(f"{SOURCE}: {len(SWEEPS)} sweeps; demo sweep {DEMO['name']}")
for k, (box, ax, pol) in enumerate(
    zip(DEMO["boxes"], DEMO["axes"], DEMO["pols"], strict=True)
):
    print(f"  box {k + 1}: (x, y, w, h) = {box}, normal along {ax}, polarity {pol:+d}")

## 2. The frame, the boxes, the Bayer binning

The camera is a colour sensor read as raw Bayer (RGGB), so neighbouring pixels
have different colour gains and a straight edge would show a 2-pixel mosaic.
The code therefore averages every **2x2 cell into one grey value** (`channel`
"sum"). One binned pixel is two sensor pixels, and binned pixel `(i, j)` sits
at sensor `(box_x + 2 j + 0.5, box_y + 2 i + 0.5)`. The box is first snapped to
even sensor coordinates so it holds whole cells.

Below: the whole frame with both boxes (display auto-levelled, as in the live
view), the raw crop of box 2, and the same crop after binning.

In [ ]:
def bin_box(frame, box, roi, axis, polarity):
    # Crop the box, bin the 2x2 cells, turn it so rows run along the edge and the
    # columns along the normal with the dark side first. Returns crop, grey, s, origin.
    bx, by, bw, bh = snap_box(box)
    rx, ry = roi
    crop = frame[by - ry : by - ry + bh, bx - rx : bx - rx + bw].astype(float)
    grey = crop.reshape(bh // 2, 2, bw // 2, 2).mean(axis=(1, 3))
    g = grey if axis == "x" else grey.T
    s = g if polarity > 0 else g[:, ::-1]
    return crop, grey, np.ascontiguousarray(s), (bx if axis == "x" else by)


frame = DEMO["frames"][4]  # the middle step, P at the calibration centre
box, axis, pol, roi = DEMO["boxes"][K], DEMO["axes"][K], DEMO["pols"][K], DEMO["roi"]
crop, grey, s, origin = bin_box(frame, box, roi, axis, pol)
print(
    "raw crop",
    crop.shape,
    "-> binned",
    grey.shape,
    "-> oriented (rows along edge)",
    s.shape,
)

if plt is not None:
    fig, axs = plt.subplots(
        1, 3, figsize=(13, 4), gridspec_kw={"width_ratios": [2.2, 1, 1]}
    )
    lo, hi = np.percentile(frame, [1, 99.5])
    extent = (roi[0], roi[0] + frame.shape[1], roi[1] + frame.shape[0], roi[1])
    axs[0].imshow(frame, cmap="gray", vmin=lo, vmax=hi, extent=extent)
    for i, b in enumerate(DEMO["boxes"]):
        bx, by, bw, bh = snap_box(b)
        colour = "C1" if i == 0 else "C3"
        axs[0].add_patch(plt.Rectangle((bx, by), bw, bh, fill=False, ec=colour, lw=1.5))
        axs[0].text(bx, by - 4, f"box {i + 1}", color=colour)
    axs[0].set_title("frame (auto-levelled), sensor px")
    axs[1].imshow(crop, cmap="gray", interpolation="nearest")
    axs[1].set_title(f"box {K + 1}: raw crop {crop.shape[1]}x{crop.shape[0]}")
    axs[2].imshow(grey, cmap="gray", interpolation="nearest")
    axs[2].set_title(f"2x2 binned {grey.shape[1]}x{grey.shape[0]}")
    fig.tight_layout()

## 3. One edge profile, two ways to read it

Inside the box the rows run along the edge and the columns across it. Take the
brightness along the columns:

- **Levels.** The dark level `low` is the median of the outer 15 % of the
  columns on the dark side, `high` the same on the bright side. Half way is
  `low + 0.5 (high - low)`.
- **`halfmax`.** In every row, the first column where the (lightly smoothed)
  brightness reaches the half-way level, interpolated linearly to a fraction of
  a column.
- **`gaussderiv`.** In every row, the column where the brightness rises
  fastest: the maximum of a Gaussian-derivative filter (width 6 sensor px =
  3 binned px, with a Gaussian of the same width along the edge to average
  noise), refined to a fraction of a column with a parabola. No levels, so it
  does not care how far the box reaches into the dark and bright sides.

The profile below is the **average of all rows after removing the edge tilt**,
so it shows the typical shape. The vertical lines are the two answers.

In [ ]:
BAND = 0.15


def levels(s):
    # [1 2 1]/4 smoothing along the normal, then the dark / bright levels.
    sm = s.copy()
    sm[:, 1:-1] = 0.25 * s[:, :-2] + 0.5 * s[:, 1:-1] + 0.25 * s[:, 2:]
    band = max(1, round(BAND * s.shape[1]))
    return sm, band, float(np.median(sm[:, :band])), float(np.median(sm[:, -band:]))


def rows_halfmax(sm, low, high):
    # Per row, the first half-way crossing (linear interpolation).
    threshold = low + 0.5 * (high - low)
    above = sm >= threshold
    first = above.argmax(axis=1)
    rows = np.nonzero(above[np.r_[: sm.shape[0]], first] & (first >= 1))[0]
    i1 = first[rows]
    v0, v1 = sm[rows, i1 - 1], sm[rows, i1]
    return rows, (i1 - 1) + (threshold - v0) / (v1 - v0)


def corr(a, k, axis):
    # Correlate every column (axis 0) or row (axis 1) of a with the odd kernel k.
    rad = k.size // 2
    pad = [(0, 0), (0, 0)]
    pad[axis] = (rad, rad)
    p = np.pad(a, pad, mode="edge")
    n = a.shape[axis]
    out = np.zeros(a.shape)
    for m in range(k.size):
        out += k[m] * (p[m : m + n] if axis == 0 else p[:, m : m + n])
    return out


def gauss_kernels(sigma):
    # Gaussian (sums to 1) and its derivative (response 1 on a unit ramp).
    rad = max(1, math.ceil(4.0 * sigma))
    x = np.r_[-rad : rad + 1].astype(float)
    g = np.exp(-0.5 * (x / sigma) ** 2)
    g /= g.sum()
    return g, x * g / sigma**2


SMOOTH = {
    "sobel": [1.0, 2.0, 1.0],
    "prewitt": [1.0, 1.0, 1.0],
    "scharr": [3.0, 10.0, 3.0],
}


def gradient(s, kind, sigma_bin=SIGMA_BIN):
    # Edge response across the columns: positive where the brightness rises.
    if kind == "gaussderiv":
        g, d = gauss_kernels(sigma_bin)
        return corr(corr(s, g, 0), d, 1)
    k = np.array(SMOOTH[kind])  # 3x3: smooth along the edge, central difference across
    return corr(corr(s, k / k.sum(), 0), np.array([-0.5, 0.0, 0.5]), 1)


def peak_position(g):
    # Parabolic sub-column maximum of each row of g (same rule as the code).
    n_rows, n_cols = g.shape
    j = 2 + g[:, 2 : n_cols - 2].argmax(axis=1)
    ar = np.r_[:n_rows]
    gm, g0, gp = g[ar, j - 1], g[ar, j], g[ar, j + 1]
    den = gm - 2.0 * g0 + gp
    with np.errstate(divide="ignore", invalid="ignore"):
        d = np.where(den < 0, 0.5 * (gm - gp) / den, 0.0)
    return j + np.clip(d, -1.0, 1.0), g0


def rows_gradpeak(s, kind, sigma_bin=SIGMA_BIN):
    # Per row, the sub-column position of the strongest rise.
    pos, peak = peak_position(gradient(s, kind, sigma_bin))
    rows = np.nonzero(peak > 0.25 * np.median(peak))[0]
    return rows, pos[rows]


def robust_line(rows, cross, n_rows):
    # Straight line c(r) = a + b r through the per-row points; outliers are dropped.
    r = rows.astype(float) - (n_rows - 0.5) / 2.0
    keep = np.ones(r.size, dtype=bool)
    for k in range(4):
        rk, ck = r[keep], cross[keep]
        rm, cm = rk.mean(), ck.mean()
        sxx = float(((rk - rm) ** 2).sum())
        b = float(((rk - rm) * (ck - cm)).sum() / sxx) if sxx > 0 else 0.0
        a = float(cm - b * rm)
        res = cross - (a + b * r)
        if k == 3:
            break
        mad = float(np.median(np.abs(res[keep] - np.median(res[keep]))))
        new_keep = np.abs(res) <= max(3.0 * 1.4826 * mad, 0.5)
        if new_keep.sum() < 3:
            break
        keep = new_keep
    return a, b


def tilt_corrected_profile(s, b):
    # Average of all rows after shifting each by the edge tilt b (columns per row).
    n_rows, n_cols = s.shape
    r = np.r_[:n_rows] - (n_rows - 0.5) / 2.0
    pos = np.clip(np.r_[:n_cols][None, :] + b * r[:, None], 0.0, n_cols - 1.0)
    i0 = np.minimum(pos.astype(int), n_cols - 2)
    frac = pos - i0
    ar = np.r_[:n_rows][:, None]
    return ((1.0 - frac) * s[ar, i0] + frac * s[ar, i0 + 1]).mean(axis=0)


def cross_level(profile, level):
    # First column position where the profile reaches level (interpolated).
    i = int(np.argmax(profile >= level))
    return (i - 1) + (level - profile[i - 1]) / (profile[i] - profile[i - 1])


def to_sensor(c):
    # Binned column index -> sensor px along the edge normal.
    return origin + 2.0 * c + 0.5


sm, band, low, high = levels(s)
rows_h, cross_h = rows_halfmax(sm, low, high)
a_h, b_h = robust_line(rows_h, cross_h, s.shape[0])
prof = tilt_corrected_profile(s, b_h)
x_sensor = to_sensor(np.r_[: prof.size])

norm = (prof - low) / (high - low)
c05, c10, c50, c90 = (cross_level(norm, v) for v in (0.05, 0.10, 0.50, 0.90))
width_1090 = 2.0 * (c90 - c10)
dprof = corr(prof[None, :], gauss_kernels(SIGMA_BIN)[1], 1)[0]
peak_c = peak_position(dprof[None, :])[0][0]
half = low + 0.5 * (high - low)
print(f"low = {low:.1f}, high = {high:.1f} counts, half-way level = {half:.1f}")
print(f"10-90 % edge width         : {width_1090:5.1f} sensor px")
print(f"50 % point (halfmax)       : {to_sensor(c50):7.1f}")
print(
    f"steepest point (gaussderiv): {to_sensor(peak_c):7.1f}   "
    f"({2 * (peak_c - c50):+.1f} px from the 50 % point)"
)
print(
    f"5 % point (what the eye sees on an auto-levelled display): {to_sensor(c05):7.1f}"
)
print(f"  -> the visible edge is {2 * (c50 - c05):.1f} px outside the 50 % cross")

if plt is not None:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(x_sensor, prof, "k.-", label="row-averaged profile")
    ax.axvspan(
        x_sensor[0],
        x_sensor[band - 1],
        color="C0",
        alpha=0.15,
        label="dark band (15 %)",
    )
    ax.axvspan(
        x_sensor[-band],
        x_sensor[-1],
        color="C1",
        alpha=0.15,
        label="bright band (15 %)",
    )
    ax.axhline(low + 0.5 * (high - low), color="C3", ls=":", label="half-way level")
    ax.axvline(to_sensor(c50), color="C3", label="halfmax")
    ax.axvline(to_sensor(peak_c), color="C2", label="gaussderiv (steepest)")
    ax.axvline(to_sensor(c05), color="grey", ls="--", label="5 % (visible edge)")
    ax2 = ax.twinx()
    ax2.plot(x_sensor, dprof, "C2--", alpha=0.7)
    ax2.set_ylabel("Gaussian-derivative response", color="C2")
    ax.set_xlabel("sensor px along the edge normal")
    ax.set_ylabel("binned brightness (counts)")
    ax.legend(loc="upper left", fontsize=8)
    fig.tight_layout()

## 4. From rows to one number and the cross

Every row gives its own edge point (dots). A **robust straight line** is fitted
through them: four passes, each dropping points further than 3 x MAD (at least
0.5 binned px) from the line, so a hot pixel or a rough row does not matter.
The line's value at the **box centre row** is the edge position; its slope is
the edge tilt, and the scatter about it is `residual_px` (how straight the edge
is; more than 3 px is refused as "edge not straight").

That position, in sensor pixels, is converted with the half-pixel rule
`position = box_start + 2 x column + 0.5` and the **cross** on the live view is
drawn there, on the line at the box-centre row. The **ring** marks the
calibration zero (where the same edge stood at best focus).

The next cell repeats the arithmetic with the code above and checks it against
the repo's `measure_edge`.

In [ ]:
n_rows = s.shape[0]
rows_g, cross_g = rows_gradpeak(s, "gaussderiv")
a_g, b_g = robust_line(rows_g, cross_g, n_rows)

for name, a_fit in (("halfmax", a_h), ("gaussderiv", a_g)):
    fix = measure_edge(
        frame,
        box,
        roi_offset=roi,
        channel="sum",
        axis=axis,
        polarity=pol,
        detector=name,
    )
    print(
        f"{name:10s}: notebook {to_sensor(a_fit):9.4f}   "
        f"measure_edge {fix.position:9.4f}   "
        f"marker (x, y) = ({fix.x:.1f}, {fix.y:.1f})   "
        f"straightness {fix.residual_px:.2f} px   edge width {fix.edge_width_px:.1f} px"
    )

if plt is not None:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.imshow(
        s,
        cmap="gray",
        interpolation="nearest",
        aspect="auto",
        extent=(origin, origin + 2 * s.shape[1], n_rows, 0),
    )
    centre = (n_rows - 0.5) / 2.0
    for rows_, cross_, a_fit, b_fit, colour, lab in (
        (rows_h, cross_h, a_h, b_h, "C3", "halfmax"),
        (rows_g, cross_g, a_g, b_g, "C2", "gaussderiv"),
    ):
        ax.plot(
            to_sensor(cross_),
            rows_ + 0.5,
            ".",
            color=colour,
            ms=4,
            label=f"{lab}: per-row points",
        )
        ax.plot(
            to_sensor(a_fit + b_fit * (np.array([0, n_rows]) - centre)),
            [0, n_rows],
            "-",
            color=colour,
            lw=1,
        )
        ax.plot([to_sensor(a_fit)], [centre + 0.5], "+", color=colour, ms=18, mew=2)
    ax.set_xlabel("sensor px along the edge normal")
    ax.set_ylabel("binned rows along the edge")
    ax.set_title("per-row points, fitted lines and the cross at the box centre")
    ax.legend(fontsize=8)
    fig.tight_layout()

## 5. Box 2 and the box width

On the rig the left edge (box 2) is wide and soft (10-90 % about 20 px, with a
bright rim and a shoulder), and the saved box was only 24-38 px wide across it.
For `halfmax` the levels come from the outer 15 % of the box on each side, so
in a narrow box the "dark" and "bright" levels are measured **on the ramp**,
the half-way level is wrong, and it moves with the edge. The measured position
then depends on how wide the box happens to be: slope and noise change with the
box.

The scan below re-measures the same frames with boxes of increasing width
across the edge (edge centred, same length along it). `gaussderiv` does not use
levels, so it hardly cares.

In [ ]:
def line_stats(p, e):
    # Slope (px/um), R2 and residual SD (px, n-2 dof) of position vs P; nan-safe.
    ok = np.isfinite(e)
    if ok.sum() < 4:
        return math.nan, math.nan, math.nan
    pp, ee = p[ok], e[ok]
    slope, icpt = np.polyfit(pp, ee, 1)
    res = ee - (icpt + slope * pp)
    r2 = 1.0 - float((res**2).sum() / ((ee - ee.mean()) ** 2).sum())
    return float(slope), r2, math.sqrt(float((res**2).sum()) / (pp.size - 2))


def inline_position(frame, box, roi, axis, polarity, detector):
    # Edge position (sensor px) with any of the five detectors; nan when it fails.
    _, _, s_, origin_ = bin_box(frame, box, roi, axis, polarity)
    sm_, band_, low_, high_ = levels(s_)
    if high_ - low_ < 10.0:
        return math.nan
    if detector == "halfmax":
        rows, cross = rows_halfmax(sm_, low_, high_)
    else:
        rows, cross = rows_gradpeak(s_, detector)
    if rows.size < max(3, 0.5 * s_.shape[0]):
        return math.nan
    a, _ = robust_line(rows, cross, s_.shape[0])
    if a < band_ - 0.5 or a > s_.shape[1] - band_ - 0.5:  # edge left the box
        return math.nan
    if polarity < 0:
        a = s_.shape[1] - 1 - a
    return origin_ + 2.0 * a + 0.5


def sweep_positions(sweep, k, detector, box=None):
    # Edge position at each P step. halfmax / gaussderiv use the repo's measure_edge.
    box = box or sweep["boxes"][k]
    axis_, pol_ = sweep["axes"][k], sweep["pols"][k]
    out = []
    for fr in sweep["frames"]:
        if detector in ("halfmax", "gaussderiv"):
            fix = measure_edge(
                fr,
                box,
                roi_offset=sweep["roi"],
                channel="sum",
                axis=axis_,
                polarity=pol_,
                detector=detector,
            )
            out.append(fix.position if fix.ok else math.nan)
        else:
            out.append(inline_position(fr, box, sweep["roi"], axis_, pol_, detector))
    return np.array(out)


def resized_box(sweep, k, normal_px, e0):
    # The same edge with another box width across it, centred on the edge.
    bx, by, bw, bh = sweep["boxes"][k]
    if sweep["axes"][k] == "x":
        return (round(e0 - normal_px / 2), by, normal_px, bh)
    return (bx, round(e0 - normal_px / 2), bw, normal_px)


e0 = float(np.nanmean(sweep_positions(DEMO, K, "halfmax")))
saved_width = DEMO["boxes"][K][2 if DEMO["axes"][K] == "x" else 3]
print(
    f"box {K + 1}: edge at {e0:.1f} sensor px; "
    f"the saved box is {saved_width} px wide across it"
)
print(
    f"{'width px':>8} | {'halfmax slope':>13} {'noise um':>9} | "
    f"{'gaussderiv slope':>16} {'noise um':>9}"
)
SCAN = {}
for width in (24, 38, 60, 90, 120):
    cells = []
    for det in ("halfmax", "gaussderiv"):
        pos = sweep_positions(DEMO, K, det, resized_box(DEMO, K, width, e0))
        sl, _, sd = line_stats(DEMO["p"], pos)
        cells.append((sl, sd / abs(sl) if sl else math.nan))
    SCAN[width] = cells
    (s_h, n_h), (s_g, n_g) = cells
    print(f"{width:8d} | {s_h:+13.3f} {n_h:9.4f} | {s_g:+16.3f} {n_g:9.4f}")

if plt is not None:
    fig, axs = plt.subplots(1, 2, figsize=(9, 3.5))
    widths = list(SCAN)
    for i, (det, colour) in enumerate((("halfmax", "C3"), ("gaussderiv", "C2"))):
        axs[0].plot(
            widths, [SCAN[w][i][0] for w in widths], "o-", color=colour, label=det
        )
        axs[1].plot(
            widths, [SCAN[w][i][1] for w in widths], "o-", color=colour, label=det
        )
    axs[0].set_ylabel("slope (px/um)")
    axs[1].set_ylabel("noise about the line (um)")
    for ax in axs:
        ax.set_xlabel("box width across the edge (sensor px)")
        ax.legend()
    fig.tight_layout()

**Reading it.** `halfmax` needs a box wide enough that both bands are on flat
ground: at least about 4x the 10-90 % width of the edge (the live panel shows
the measured width). `gaussderiv` gives nearly the same slope and noise from a
box about 2x the width upward. On the rig the same frames gave, for box 2,
slope 1.0 to 1.75 px/um and noise 0.078 to 0.033 um with `halfmax` as the box
grew from 24 to 90 px; `gaussderiv` stayed at 0.033 to 0.050 um.

## 6. Why not Sobel or Prewitt

A Sobel, Prewitt or Scharr filter is a **3x3 kernel**: smooth along the edge,
then subtract the two columns beside each pixel. The edge is then the maximum
of that difference. The idea is right (it is what `gaussderiv` also does), but
the difference spans only 3 binned columns while the ramp is 7-14 binned
pixels wide. Across a ramp the response is nearly flat at its top, so the
position of the maximum jumps about with the noise from row to row. A Gaussian
derivative of width 3 binned px matches the ramp, so its peak is sharp and
stable.

The cell below measures all five detectors on every sweep. The `halfmax` and
`gaussderiv` positions come from `measure_edge`; Sobel, Prewitt and Scharr are
written inline above (numpy only) and use the same per-row peak, the same
robust line and the same position conversion.

In [ ]:
RESULTS = {}  # (box index, detector) -> [(slope, R2, resid px, resid um), ...]
for sw in SWEEPS:
    for k in range(len(sw["boxes"])):
        for det in DETECTORS:
            sl, r2, sd = line_stats(sw["p"], sweep_positions(sw, k, det))
            RESULTS.setdefault((k, det), []).append(
                (sl, r2, sd, sd / abs(sl) if sl else math.nan)
            )


def mean_stats(k, det):
    arr = np.array(RESULTS[(k, det)])
    return np.nanmean(arr, axis=0), len(arr)


print(f"Source: {SOURCE}, {len(SWEEPS)} sweeps. Noise = scatter of the edge position")
print("about the sweep line, in um (lower is better).")
for k, label in ((0, "box 1, top edge"), (1, "box 2, left edge")):
    if (k, "halfmax") not in RESULTS:
        continue
    print(f"\n{label} (mean over {mean_stats(k, 'halfmax')[1]} sweeps)")
    cols = ("slope px/um", "R2", "resid px", "noise um", "vs halfmax")
    print(f"{'detector':>11} | " + " ".join(f"{c:>10}" for c in cols))
    base = mean_stats(k, "halfmax")[0][3]
    for det in DETECTORS:
        (sl, r2, sd, um), _ = mean_stats(k, det)
        print(
            f"{det:>11} | {sl:+11.3f} {r2:7.4f} {sd:9.3f} {um:9.4f} {um / base:9.2f}x"
        )

if plt is not None:
    fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
    ks = [k for k in (0, 1) if (k, "halfmax") in RESULTS]
    bar = 0.8 / max(len(ks), 1)
    for n, k in enumerate(ks):
        heights = [mean_stats(k, d)[0][3] for d in DETECTORS]
        axs[0].bar(
            np.r_[: len(DETECTORS)] + n * bar, heights, bar, label=f"box {k + 1}"
        )
    axs[0].set_xticks(np.r_[: len(DETECTORS)] + bar * (len(ks) - 1) / 2, DETECTORS)
    axs[0].set_ylabel("noise about the sweep line (um), lower is better")
    axs[0].legend()
    for det in DETECTORS:
        pos = sweep_positions(DEMO, K, det)
        sl, _, _ = line_stats(DEMO["p"], pos)
        icpt = np.nanmean(pos) - sl * np.nanmean(DEMO["p"])
        axs[1].plot(
            DEMO["p"], (pos - (icpt + sl * DEMO["p"])) / abs(sl), "o-", ms=3, label=det
        )
    axs[1].set_xlabel("piezo P - centre (um)")
    axs[1].set_ylabel("departure from the line (um)")
    axs[1].set_title(f"box {K + 1}, sweep {DEMO['name']}")
    axs[1].legend(fontsize=8)
    fig.tight_layout()

**Conclusion (from the study of the rig sweeps on 2026-10-28; mean of the
recorded sweeps).**

| detector     | box 1 noise (um) | box 2 noise (um) |
| ------------ | ---------------- | ---------------- |
| `halfmax`    | 0.033            | 0.075            |
| Sobel        | 0.050            | 0.143            |
| Prewitt      | 0.051            | 0.122            |
| Scharr       | 0.053            | 0.160            |
| `gaussderiv` | 0.028            | 0.039            |

Sobel, Prewitt and Scharr were 1.5 to 2x noisier on the real residual and 3 to
6x noisier when pixel noise was injected into the same frames, in every sweep.
`gaussderiv` is as good as `halfmax` on the clean top edge and twice as good on
the soft left edge (0.039 against 0.075 um), with a slope that does not depend
on the box width. Fusing both edges gave 0.021 um with `gaussderiv` against
0.023 um with `halfmax`. So `gaussderiv` (width 6 sensor px) is the default,
`halfmax` stays selectable, and the three classic filters are not used. The
numbers printed above should agree within sweep-to-sweep variation.

## 7. What "noise in um" means, and how it weights the two edges

An edge's position in pixels is converted to a focus error with the slope of
its own calibration: `error_um = (position - zero) / slope`. Pixel scatter
converts the same way:

`noise_um = max(noise_px, residual_px, 0.02) / |slope px per um|`

so a flatter slope (the same pixel scatter is more microns) or a noisier edge
means a larger noise in um. When both edges are used they are averaged with
weights `1 / noise_um^2` (inverse variance), and the fused noise is
`1 / sqrt(w1 + w2)`, always below the better edge alone. If the two edges
disagree by more than the limit the loop does not average them at all (see the
doc, section "Optional second edge and fusion").

In [ ]:
print(f"Sweep {DEMO['name']} ({SOURCE})")
for det in ("halfmax", "gaussderiv"):
    noises = []
    for k in range(len(DEMO["boxes"])):
        sl, _, sd = line_stats(DEMO["p"], sweep_positions(DEMO, k, det))
        noises.append(max(sd, 0.02) / abs(sl))
        print(
            f"  {det:10s} box {k + 1}: slope {sl:+.2f} px/um, "
            f"residual {sd:.3f} px -> noise {noises[-1]:.3f} um"
        )
    if len(noises) == 2:
        w = 1.0 / np.square(noises)
        share = w / w.sum()
        print(
            f"  {det:10s} weights {share[0]:.0%} / {share[1]:.0%}; "
            f"fused noise {1.0 / math.sqrt(w.sum()):.3f} um"
        )

## Summary

- The cross is where a **straight line through one edge point per row**
  crosses the box centre row; with `halfmax` each point is the half-way
  brightness crossing, with `gaussderiv` it is the steepest rise.
- An auto-levelled display shows the edge at about the 5 % point, which sits
  10-14 px outside the 50 % cross; that is why the cross can look "inside" the
  edge. Only changes of the position matter for focus, not where it sits.
- A narrow box biases `halfmax` (levels measured on the ramp); `gaussderiv` is
  insensitive to it.
- Sobel, Prewitt and Scharr are noisier than both; they are not used.
- A calibration is valid only for the detector it was made with. The rig's
  existing calibration is `halfmax`: recalibrate with `gaussderiv`, or set the
  detector to `halfmax`.